In [102]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
import requests
import math
import os
import json
from tavily import TavilyClient
load_dotenv()
llm = ChatGroq(model="qwen/qwen3.6-27b", temperature=0.5, max_tokens=500)
clint = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
@tool
def search_tool(query:str)->str:
    """Expart in extrxt information about the topic provided"""

    result = clint.search(
        max_results=5,
       
        topic="general",
        search_depth="advanced",
        query=query)

    return json.dumps(result)

@tool
def Calculator(expression:str)->str:
    """Usful for simple math calculation Input should be a  valid math expression . 
    Example:2+2,math.sqrt(16),10*5"""

    try:
        allowed = {
            "math":math,
            "abs":abs,
            "round":round,
            "min":min,
            "max":max,
            "sum":sum
        }
        result = eval(expression,{"__builtines__":{}},allowed)
        return str(result)
    except expression as e:
        print(e)
import os
import requests
@tool
def get_stock_price(symbol: str) -> dict:
    """Fetch latest stock price for a given ticker symbol using Alpha Vantage."""
    api_key = os.getenv("ALPHA_VANTAGE_API_KEY")
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={api_key}"
    response = requests.get(url)
    return response.json()
import requests
from dotenv import load_dotenv
import os

load_dotenv()
@tool
def get_current_weather(city,units="metric"):
   
    """
    Fetch current weather for a given city using OpenWeatherMap API.

    Args:
        city (str): City name, e.g. "Dhaka" or "Dhaka,BD"
        units (str): "metric" (Celsius), "imperial" (Fahrenheit), or "standard" (Kelvin)

    Returns:
        dict: Parsed weather data, or None if the request failed
    """

    api_key = os.getenv("OPENWEATHER_API_KEY")
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city,
        "appid": api_key,
        "units": units
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        return {
            "city": data["name"],
            "country": data["sys"]["country"],
            "temperature": data["main"]["temp"],
            "feels_like": data["main"]["feels_like"],
            "humidity": data["main"]["humidity"],
            "pressure": data["main"]["pressure"],
            "weather": data["weather"][0]["description"],
            "wind_speed": data["wind"]["speed"]
        }

    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    except KeyError as e:
        print(f"Unexpected response format, missing key: {e}")

    return None

tools = [search_tool,Calculator,get_stock_price,get_current_weather]
llm_with_tools = llm.bind_tools(tools=tools)
class chatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
def chatbot(state: chatState):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)    
graph = StateGraph(chatState)
graph.add_node("chatbot",chatbot)
graph.add_node("tools",tool_node)

graph.add_edge(START,"chatbot")
graph.add_conditional_edges("chatbot",tools_condition)
graph.add_edge("tools","chatbot")
graph.add_edge("chatbot",END)

wf = graph.compile()
wf
init = {
    "messages":[HumanMessage(content="what is the stoke price in BMW? today")]
}
result = wf.invoke(init)
print(result['messages'][-1].content)

while True:
    print("Press EXIT to return.")
    user_input = input("Enter the query:")
    if user_input.lower()=="exit":
        print("Good bye")
        break
    else:
        init = {
            "messages":user_input
        }
        result = wf.invoke(init)
        print("AI:",result["messages"][-1].content)    



Press EXIT to return.
AI: The current weather in Dhaka is overcast clouds with a temperature of approximately **28°C** (feels like 34°C). The humidity is quite high at 89%, and the wind speed is around 1.54 m/s.
Press EXIT to return.


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.6-27b` in organization `org_01kqvjnfv9etbbn09za3h7f9zh` service tier `on_demand` on output tokens per minute (OTPM): Limit 1000, Used 524, Requested 500. Please try again in 1.44s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}